In [12]:
# Cell 1: Imports and setup
# Loads libraries, defines all constants in one place.
# RANDOM_SEED is fixed at 524 forever — never change this.

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

RANDOM_SEED = 524
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

INPUT_PATH = "/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/lambda calculation/hinemo_dataset_with_lambda.csv"
SPLITS_DIR = "/Users/harshaggarwal/Projects_4/hinemo_project/data/splits"

os.makedirs(SPLITS_DIR, exist_ok=True)
print(f"Output directory ready: {SPLITS_DIR}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Split ratio: {TRAIN_RATIO} / {VAL_RATIO} / {TEST_RATIO}")

Output directory ready: /Users/harshaggarwal/Projects_4/hinemo_project/data/splits
Random seed: 524
Split ratio: 0.7 / 0.15 / 0.15


In [13]:
# Cell 2: Load the full dataset and print basic stats.
# This is just a sanity check — confirm shape, columns,
# emotion distribution, and null counts before touching anything.

df = pd.read_csv(INPUT_PATH)
print(f"Full dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

print(f"\nEmotion distribution (full dataset):")
counts = df["gpt_emotion"].value_counts()
pcts   = df["gpt_emotion"].value_counts(normalize=True) * 100
print(pd.DataFrame({"count": counts, "pct": pcts.round(1)}))

print(f"\nNull values per column:")
print(df.isnull().sum())

Full dataset shape: (46174, 9)
Columns: ['id', 'text', 'gpt_emotion', 'source', 'indiclid_label', 'indiclid_confidence', 'indiclid_score', 'indiclid_model_used', 'lambda']

Emotion distribution (full dataset):
             count   pct
gpt_emotion             
joy          17715  38.4
anger        12369  26.8
disgust       8698  18.8
sadness       7392  16.0

Null values per column:
id                     0
text                   0
gpt_emotion            0
source                 0
indiclid_label         0
indiclid_confidence    0
indiclid_score         0
indiclid_model_used    0
lambda                 0
dtype: int64


In [14]:
# Cell 3: Check for duplicates BEFORE splitting.
# If the same comment appears in both train and test,
# the model has effectively "seen" test data during training
# — which inflates test scores and makes results unreliable.
# We check both ID and text separately.

print("=" * 60)
print("PRE-SPLIT DUPLICATE CHECK")
print("=" * 60)

id_dupes   = df["id"].duplicated().sum()
text_dupes = df["text"].duplicated().sum()

print(f"Duplicate IDs:   {id_dupes}")
print(f"Duplicate texts: {text_dupes}")

if id_dupes > 0 or text_dupes > 0:
    print("\n[!] Duplicates found — dropping before splitting")
    df = df.drop_duplicates(subset=["id"],   keep="first")
    df = df.drop_duplicates(subset=["text"], keep="first")
    print(f"Rows after dedup: {len(df)}")
else:
    print("\nNo duplicates found — dataset is clean, proceeding.")

PRE-SPLIT DUPLICATE CHECK
Duplicate IDs:   0
Duplicate texts: 1

[!] Duplicates found — dropping before splitting
Rows after dedup: 46173


In [15]:
# Cell 3b: Drop text-level duplicates from the full dataset
# before splitting. These are comments with identical text
# but different source_ids — different people who independently
# posted the same short phrase. Keeping them risks the same
# text appearing in both train and test (data leakage).
# We already checked ID-level duplicates in Cell 3.
# This cell handles content-level duplicates.

before = len(df)
df = df.drop_duplicates(subset=["text"], keep="first")
after = len(df)

print(f"Rows before text dedup: {before:,}")
print(f"Rows after text dedup:  {after:,}")
print(f"Dropped:                {before - after:,}")
print(f"\nEmotion distribution after text dedup:")
counts = df["gpt_emotion"].value_counts()
pcts   = df["gpt_emotion"].value_counts(normalize=True) * 100
print(pd.DataFrame({"count": counts, "pct": pcts.round(1)}))

Rows before text dedup: 46,173
Rows after text dedup:  46,173
Dropped:                0

Emotion distribution after text dedup:
             count   pct
gpt_emotion             
joy          17715  38.4
anger        12368  26.8
disgust       8698  18.8
sadness       7392  16.0


In [16]:
# Cell 4: Create train / val / test splits.
# Strategy: two-step stratified split.
#   Step 1 — split full dataset into train (70%) and temp (30%)
#   Step 2 — split temp 50/50 into val (15%) and test (15%)
# Stratified by gpt_emotion so every emotion class keeps
# its original proportion in all three splits.

# Step 1: train vs temp
train_df, temp_df = train_test_split(
    df,
    test_size=1 - TRAIN_RATIO,     # 30% goes to temp
    stratify=df["gpt_emotion"],
    random_state=RANDOM_SEED,
)

# Step 2: val vs test (split temp 50/50)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,                  # 50% of temp = 15% of total
    stratify=temp_df["gpt_emotion"],
    random_state=RANDOM_SEED,
)

print(f"Train : {len(train_df):,} rows ({len(train_df)/len(df):.1%})")
print(f"Val   : {len(val_df):,} rows ({len(val_df)/len(df):.1%})")
print(f"Test  : {len(test_df):,} rows ({len(test_df)/len(df):.1%})")
print(f"Total : {len(train_df) + len(val_df) + len(test_df):,} rows")

Train : 32,321 rows (70.0%)
Val   : 6,926 rows (15.0%)
Test  : 6,926 rows (15.0%)
Total : 46,173 rows


In [17]:
# Cell 5: Confirm stratification worked correctly.
# Every split should have roughly the same emotion proportions
# as the full dataset. If any split looks wildly different,
# something went wrong with the stratification.

print("=" * 60)
print("EMOTION DISTRIBUTION PER SPLIT")
print("=" * 60)

for name, split in [("Full", df), ("Train", train_df),
                    ("Val", val_df), ("Test", test_df)]:
    counts = split["gpt_emotion"].value_counts()
    pcts   = split["gpt_emotion"].value_counts(normalize=True) * 100
    print(f"\n{name} ({len(split):,} rows):")
    for emotion in ["anger", "disgust", "joy", "sadness"]:
        c = counts.get(emotion, 0)
        p = pcts.get(emotion, 0)
        print(f"  {emotion:<10} {c:>6,}  ({p:.1f}%)")

EMOTION DISTRIBUTION PER SPLIT

Full (46,173 rows):
  anger      12,368  (26.8%)
  disgust     8,698  (18.8%)
  joy        17,715  (38.4%)
  sadness     7,392  (16.0%)

Train (32,321 rows):
  anger       8,658  (26.8%)
  disgust     6,089  (18.8%)
  joy        12,400  (38.4%)
  sadness     5,174  (16.0%)

Val (6,926 rows):
  anger       1,855  (26.8%)
  disgust     1,304  (18.8%)
  joy         2,658  (38.4%)
  sadness     1,109  (16.0%)

Test (6,926 rows):
  anger       1,855  (26.8%)
  disgust     1,305  (18.8%)
  joy         2,657  (38.4%)
  sadness     1,109  (16.0%)


In [18]:
# Cell 6: Verify there is ZERO overlap between splits.
# A comment appearing in both train and test is data leakage.
# We check by ID (exact match) and by text (content match).
# Both should be 0. If either is non-zero, something is wrong.

print("=" * 60)
print("POST-SPLIT CROSS-CONTAMINATION CHECK")
print("=" * 60)

train_ids   = set(train_df["id"].astype(str))
val_ids     = set(val_df["id"].astype(str))
test_ids    = set(test_df["id"].astype(str))

train_texts = set(train_df["text"].str.strip().str.lower())
val_texts   = set(val_df["text"].str.strip().str.lower())
test_texts  = set(test_df["text"].str.strip().str.lower())

# Calculate overlaps
id_train_val   = train_ids & val_ids
id_train_test  = train_ids & test_ids
id_val_test    = val_ids & test_ids

text_train_val  = train_texts & val_texts
text_train_test = train_texts & test_texts
text_val_test   = val_texts & test_texts

print(f"ID overlap — train ∩ val  : {len(id_train_val)}")
print(f"ID overlap — train ∩ test : {len(id_train_test)}")
print(f"ID overlap — val ∩ test   : {len(id_val_test)}")

print(f"\nText overlap — train ∩ val  : {len(text_train_val)}")
print(f"Text overlap — train ∩ test : {len(text_train_test)}")
print(f"Text overlap — val ∩ test   : {len(text_val_test)}")

all_clean = (
    len(id_train_val) == 0 and
    len(id_train_test) == 0 and
    len(id_val_test) == 0 and
    len(text_train_val) == 0 and
    len(text_train_test) == 0 and
    len(text_val_test) == 0
)

print(f"\n{'✅ All checks passed — zero overlap across splits.' if all_clean else '❌ OVERLAP DETECTED — investigate before proceeding.'}")

# --------------------------------------------------------
# Display the actual overlapping IDs/texts
# --------------------------------------------------------
if not all_clean:
    print("\n" + "=" * 60)
    print("OVERLAPPING IDs")
    print("=" * 60)

    print("\nTrain ∩ Val IDs:")
    print(sorted(id_train_val))

    print("\nTrain ∩ Test IDs:")
    print(sorted(id_train_test))

    print("\nVal ∩ Test IDs:")
    print(sorted(id_val_test))

    print("\n" + "=" * 60)
    print("OVERLAPPING TEXTS")
    print("=" * 60)

    print("\nTrain ∩ Val Texts:")
    for t in sorted(text_train_val):
        print("-", t)

    print("\nTrain ∩ Test Texts:")
    for t in sorted(text_train_test):
        print("-", t)

    print("\nVal ∩ Test Texts:")
    for t in sorted(text_val_test):
        print("-", t)

POST-SPLIT CROSS-CONTAMINATION CHECK
ID overlap — train ∩ val  : 0
ID overlap — train ∩ test : 0
ID overlap — val ∩ test   : 0

Text overlap — train ∩ val  : 22
Text overlap — train ∩ test : 20
Text overlap — val ∩ test   : 2

❌ OVERLAP DETECTED — investigate before proceeding.

OVERLAPPING IDs

Train ∩ Val IDs:
[]

Train ∩ Test IDs:
[]

Val ∩ Test IDs:
[]

OVERLAPPING TEXTS

Train ∩ Val Texts:
- bharatiya ekta zindabad ❤
- bharatiya ekta zindabad 🇮🇳
- bhartiya ekta zindabad ❤❤
- bhartiya ekta zindabad ❤❤❤
- bhartiya ekta zindabad 🇮🇳 ❤
- bhartiya ekta zindabad 🇮🇳🇮🇳
- bhartiya ekta zindabad 🫡
- happy diwali nick and carrie
- i love my india
- i miss you dhoni
- i miss you ms sir
- i miss you sushant sir
- love you msd sir
- miss you dhoni sir 😭
- miss you ms dhoni
- miss you susant sir
- miss you sushant bhai
- miss you sushant singh rajput 😭😭😭😭😭
- support for sonam wangchuk sir 😢
- we miss you sir 😢😢
- we support sonam sir
- we support sonam sir ❤

Train ∩ Test Texts:
- bharat mata ki 

In [19]:
# Cell 7: Check that λ (Hindi Lexical Dominance Score)
# is similarly distributed across all three splits.
# If train has much higher mean λ than test, the probing
# experiments could be biased — the model sees more
# Hindi-dominant text during training than at test time.

print("=" * 60)
print("λ DISTRIBUTION PER SPLIT")
print("=" * 60)

for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n{name}:")
    print(f"  mean λ : {split['lambda'].mean():.4f}")
    print(f"  std λ  : {split['lambda'].std():.4f}")
    print(f"  median : {split['lambda'].median():.4f}")
    print(f"  λ=0    : {(split['lambda']==0).sum():,} ({(split['lambda']==0).mean():.1%})")

λ DISTRIBUTION PER SPLIT

Train:
  mean λ : 0.4079
  std λ  : 0.2827
  median : 0.4615
  λ=0    : 6,173 (19.1%)

Val:
  mean λ : 0.4095
  std λ  : 0.2838
  median : 0.4615
  λ=0    : 1,312 (18.9%)

Test:
  mean λ : 0.4043
  std λ  : 0.2825
  median : 0.4545
  λ=0    : 1,346 (19.4%)


In [20]:
# Cell 8: Save all three splits to data/splits/.
# These files are FINAL — do not modify after this point.
# All downstream steps (fine-tuning, probing, CKA, SHAP)
# use exactly these files. The random seed 524 ensures
# anyone can reproduce these exact splits.

train_path = os.path.join(SPLITS_DIR, "hinemo_train.csv")
val_path   = os.path.join(SPLITS_DIR, "hinemo_val.csv")
test_path  = os.path.join(SPLITS_DIR, "hinemo_test.csv")

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path,     index=False)
test_df.to_csv(test_path,   index=False)

print("=" * 60)
print("SPLITS SAVED")
print("=" * 60)
print(f"Train : {train_path}")
print(f"        {len(train_df):,} rows")
print(f"Val   : {val_path}")
print(f"        {len(val_df):,} rows")
print(f"Test  : {test_path}")
print(f"        {len(test_df):,} rows")
print(f"\nRandom seed used: {RANDOM_SEED}")
print("These splits are locked. Do not regenerate.")

SPLITS SAVED
Train : /Users/harshaggarwal/Projects_4/hinemo_project/data/splits/hinemo_train.csv
        32,321 rows
Val   : /Users/harshaggarwal/Projects_4/hinemo_project/data/splits/hinemo_val.csv
        6,926 rows
Test  : /Users/harshaggarwal/Projects_4/hinemo_project/data/splits/hinemo_test.csv
        6,926 rows

Random seed used: 524
These splits are locked. Do not regenerate.
